# Hala H22 Ã¢â‚¬â€ prospective TEST batch replication

One free Kaggle Save & Run All version. No training, no paid compute, no production serving and no change to previous published outcomes. The 2026-10-17 earlier FP16 TEST stays independent; prior INT8 is unknown. The actual assigned T4 is bound on the batch worker before inference. Saved outputs require later retrieval, hashes, independent audits and publication. This notebook is NOT release qualification.


In [ ]:
import os
from pathlib import Path
assert os.name == 'posix' and Path('/kaggle/working').is_dir()
assert os.environ.get('KAGGLE_KERNEL_RUN_TYPE') == 'Batch'
assert not Path('/kaggle/working/hala-h22-batch').exists(), 'Do not overwrite a prior run'
# No model weights downloaded or GPU model allocated in this cell.


Qualify all pinned packages before model allocation. No paid plan, token, service credentials, quotas bypass or auto-upgrade. Installed Torch must already match the reviewed GPU environment.


In [ ]:
%pip install --quiet --no-deps transformers==5.16.1 accelerate==1.12.0 bitsandbytes==0.50.2 peft==0.20.0 safetensors==0.8.0 Pillow==11.3.0
import importlib.metadata
assert importlib.metadata.version('torch') == '2.11.0+cu128'


In [ ]:
import hashlib, json, subprocess, sys
from pathlib import Path
data = "\"\"\"Prospective H22 free-Kaggle Save & Run All batch operator.\n\nThis operator does not train, tune, merge, deploy or publish model artifacts.\nIt is deliberately distinct from the historical interactive attempt3 operator.\nAll three TEST modes must run with the same observed physical GPU in ONE batch.\n\"\"\"\nimport hashlib\nimport importlib.metadata\nimport io\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport time\nimport urllib.request\nimport zipfile\n\nSOURCE = \"07591f109b6683c9302fcf890259ddf0bde91494\"\nFREEZE = \"98ca7bba6fc16dd6cd737eaad6de1e425e30aef67116c379db36fbbb85b2af4b\"\nMODEL_REV = \"89644892e4d85e24eaac8bacfd4f463576704203\"\nARTIFACT_COMMIT = \"c7f2e16244910c7b4246749b3186d49fca5ce7b4\"\nDEV_SHA = \"60a0986cc196041c2474314624e0f88c9c2986d75bc58cf0ddb44623ed294e02\"\nH19_SHA = \"d6be1ffba54a3a44978da4c922130fc2d7e74bdc2532a14d0d1e7cc3a691cead\"\nH19_MANIFEST = \"ced87ae6874df6de2ed1ab853312824084191a29a42b97869f6eb462aceb629e\"\nADAPTER_MANIFEST = \"7e46995aa694e5782a93d4beb6139cc5880b898a2038eb2d2ce2a2d8006a841d\"\nEARLIER_FP16_ARCHIVE = \"e098a7631708aa50609e56ba7701c01a1c5cc5804b87f9994b6dded181abf471\"\nEARLIER_FP16_COMMIT = \"9568d58c74f3e1326b55b04d304f643e2bf1b99f\"\nARCHIVE_BASE = \"https://raw.githubusercontent.com/TheHalfMoon/Hala/\"\nMODES = (\"fp16\", \"int8\", \"nf4\")\nROOT = Path(\"/kaggle/working/hala-h22-batch\")\nCACHE = \"/tmp/hala-h22-cache\"\nADAPTER_FILES = {\n    \"adapter_config.json\": \"21bbc4037d9fe27e1281700e72dcc402c4ce801d2d27798ac64cba33ac4d5de2\",\n    \"adapter_model.safetensors\": \"7eb2f17df11151e5b77a5c715c59f56c854ca7b0d0da3ee591f394b501826fa7\",\n    \"training_metadata.json\": \"d1251b39c3d71fcd410ec0fce631b2c065a6df4978309bedaff168807314020d\",\n    \"manifest.json\": ADAPTER_MANIFEST,\n}\nADAPTER_IDENTITY = {\n    \"base_model\": \"Qwen/Qwen3-VL-2B-Instruct\", \"base_revision\": MODEL_REV,\n    \"data_sha256\": \"af540e4db2289540fb8adecd0665439c4d6055ffc86673a3286e902c86c0340e\",\n    \"evaluation_sha256\": \"e9f98694089d5411d3a6c570c7de2ede8f82cbeb3132e47904d17301030c0768\",\n    \"source_commit\": \"5b6eb4bf29acbaf21aa199f113f78af4f268b3b7\",\n}\nDEV = (\n    (\"e252b7805055fa6a89e1d92b6bba7faa5dbf1efc\", \"6d5cfe7c3746a76eaceb8ebc37605d0d98cc69b69ca94765421f081b64d64580\"),\n    (\"e3d400424766aad3c5856ba29266e11033392ef5\", \"e0dbe2d692c522943b3e0c49c198e540416cbabfa2b19499c9da4a518b8339e5\"),\n    (\"d08f522de433d9da866ec3907161e4a9b326efba\", \"84dac2fd6ab44d751cefbe9ce61a958ef44a39aad3b1f09e880c5350bbd87458\"),\n)\n\n\ndef digest(raw):\n    return hashlib.sha256(raw).hexdigest()\n\n\ndef atomic_json(path, payload):\n    path = Path(path)\n    temp = path.with_suffix(path.suffix + \".partial\")\n    with temp.open(\"w\", encoding=\"utf-8\") as handle:\n        json.dump(payload, handle, sort_keys=True, allow_nan=False)\n        handle.write(\"\\n\")\n        handle.flush()\n        os.fsync(handle.fileno())\n    temp.replace(path)\n\n\ndef public_artifact(commit, suffix, expected, maximum):\n    \"\"\"Read only immutable public artifacts; bounds and digest are mandatory.\"\"\"\n    if re.fullmatch(r\"[a-f0-9]{40}\", commit) is None:\n        raise ValueError(\"Immutable commit SHA required\")\n    if not suffix or suffix.startswith(\"/\") or \"..\" in Path(suffix).parts:\n        raise ValueError(\"Unsafe artifact path\")\n    url = ARCHIVE_BASE + commit + \"/\" + suffix\n    with urllib.request.urlopen(url, timeout=45) as reply:\n        if reply.geturl() != url:\n            raise ValueError(\"Unexpected artifact redirect\")\n        raw = reply.read(maximum + 1)\n    if len(raw) > maximum or digest(raw) != expected:\n        raise ValueError(\"Public evidence digest or size mismatch: \" + suffix)\n    return raw\n\n\ndef safe_unpack(raw, target):\n    \"\"\"No traversal, links, duplicate members, ZIP bombs or replacement.\"\"\"\n    target = Path(target)\n    if target.exists():\n        raise FileExistsError(\"Existing evidence/dataset directory must not be replaced\")\n    target.mkdir(parents=True)\n    with zipfile.ZipFile(io.BytesIO(raw)) as z:\n        infos = z.infolist()\n        if len(infos) > 512 or len({x.filename for x in infos}) != len(infos):\n            raise ValueError(\"Unexpected ZIP member count or duplicates\")\n        if sum(x.file_size for x in infos) > 32 * 1024**2:\n            raise ValueError(\"ZIP content budget exceeded\")\n        for item in infos:\n            name = item.filename\n            if (not name or name.startswith(\"/\") or chr(92) in name or\n                    any(x in (\"\", \".\", \"..\") for x in name.rstrip(\"/\").split(\"/\")) or\n                    item.file_size > 8 * 1024**2 or ((item.external_attr >> 16) & 0o170000) == 0o120000):\n                raise ValueError(\"Unsafe ZIP member\")\n            path = target / name\n            if item.is_dir():\n                path.mkdir(parents=True, exist_ok=True)\n                continue\n            path.parent.mkdir(parents=True, exist_ok=True)\n            with z.open(item) as src, path.open(\"xb\") as dst:\n                shutil.copyfileobj(src, dst, length=1024 * 1024)\n\n\ndef prepare(root=ROOT):\n    \"\"\"Network-only source/evidence preparation. No model download.\"\"\"\n    if os.name != \"posix\" or not Path(\"/kaggle/working\").is_dir():\n        raise RuntimeError(\"Must run in actual remote Kaggle Linux environment\")\n    if root.exists():\n        raise FileExistsError(\"Batch workspace already exists; never overwrite old evidence\")\n    root.mkdir(parents=True)\n    source = root / \"source\"\n    subprocess.run([\"git\", \"clone\", \"--quiet\", \"https://github.com/TheHalfMoon/Hala.git\", str(source)],\n                   check=True, timeout=240)\n    subprocess.run([\"git\", \"checkout\", \"--detach\", SOURCE], cwd=source, check=True, timeout=60,\n                   stdout=subprocess.DEVNULL)\n    if subprocess.check_output([\"git\", \"rev-parse\", \"HEAD\"], cwd=source, text=True).strip() != SOURCE:\n        raise ValueError(\"Historical source identity mismatch\")\n    if subprocess.check_output([\"git\", \"status\", \"--porcelain\", \"--untracked-files=all\"], cwd=source, text=True).strip():\n        raise ValueError(\"Historical source must be clean\")\n    safe_unpack(public_artifact(ARTIFACT_COMMIT, \"h19-freeze/hala-h21-frozen.zip\", H19_SHA, 3 * 1024**2),\n                root / \"frozen\")\n    adapter = root / \"retained-adapter\"\n    adapter.mkdir()\n    for name, expected in ADAPTER_FILES.items():\n        bound = 4 * 1024**2 if name.endswith(\"safetensors\") else 65536\n        (adapter / name).write_bytes(public_artifact(ARTIFACT_COMMIT,\n            \"h21-shared-pilot/trained-adapter/\" + name, expected, bound))\n    sys.path.insert(0, str(source))\n    from development.freeze import verify_bundle\n    from training.adapter_archive import verify_bundle as verify_adapter\n    verify_bundle(root / \"frozen\", require_media=True, expected_manifest_sha256=H19_MANIFEST)\n    verify_adapter(adapter, ADAPTER_IDENTITY, ADAPTER_MANIFEST)\n    # Immutable external evidence is verified before rerunning old TEST cases.\n    for n, (commit, expected) in enumerate(DEV):\n        mode = MODES[n]\n        raw = public_artifact(commit, f\"h22-attempt2/profile-{n}-{mode}-dev-evidence.zip\",\n                              expected, 16 * 1024**2)\n        with zipfile.ZipFile(io.BytesIO(raw)) as z:\n            if digest(z.read(\"experiment.json\")) != DEV_SHA:\n                raise ValueError(\"Retained DEV experiment hash mismatch\")\n    previous = public_artifact(EARLIER_FP16_COMMIT,\n        \"h22-test-restart/profile-0-fp16-test-evidence.zip\", EARLIER_FP16_ARCHIVE, 16 * 1024**2)\n    with zipfile.ZipFile(io.BytesIO(previous)) as z:\n        last = json.loads(z.read(\"profile-status.json\"))\n        if last.get(\"index\") != 0 or last.get(\"status\") != \"finished\":\n            raise ValueError(\"Earlier TEST evidence is not a completed FP16 run\")\n    return source\n\n\ndef qualify(source, output):\n    os.environ[\"HALA_REMOTE_DEVELOPMENT_ALLOWED\"] = \"1\"\n    os.environ[\"HF_HUB_DISABLE_TELEMETRY\"] = \"1\"\n    os.environ[\"DO_NOT_TRACK\"] = \"1\"\n    for key in (\"HF_HOME\", \"HF_HUB_CACHE\", \"HUGGINGFACE_HUB_CACHE\",\n                \"TRANSFORMERS_CACHE\", \"XDG_CACHE_HOME\", \"TORCH_HOME\"):\n        if key in os.environ and Path(os.environ[key]).resolve() != Path(CACHE).resolve():\n            raise ValueError(\"Unreviewed cache environment override\")\n        os.environ[key] = CACHE\n    from evaluation import development_precision as precision\n    from evaluation.development_runtime import evaluation_guard\n    from development.media import FONT_SHA256\n    from PIL import features\n    evaluation_guard(\"kaggle\")\n    precision.verify_sources(FREEZE)\n    precision.check_cache(CACHE)\n    for package, expected in {\"torch\": \"2.11.0+cu128\", \"transformers\": \"5.16.1\",\n             \"Pillow\": \"11.3.0\", \"accelerate\": \"1.12.0\", \"bitsandbytes\": \"0.50.2\",\n             \"safetensors\": \"0.8.0\", \"peft\": \"0.20.0\"}.items():\n        if importlib.metadata.version(package) != expected:\n            raise RuntimeError(\"Unqualified environment package: \" + package)\n    import torch\n    if not torch.cuda.is_available() or torch.cuda.get_device_name(0) != \"Tesla T4\":\n        raise RuntimeError(\"Actual free Tesla T4 required\")\n    uuid = precision.resolve_device_uuid(torch)\n    if features.version_feature(\"raqm\") != \"0.10.1\":\n        raise RuntimeError(\"RAQM rendering mismatch\")\n    font = Path(\"/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf\")\n    if digest(font.read_bytes()) != FONT_SHA256:\n        raise RuntimeError(\"Font content mismatch\")\n    report = {\"schema\": \"hala.h22.batch_assigned_hardware.v1\",\n              \"provider\": \"kaggle\", \"kernel_run_type\": os.getenv(\"KAGGLE_KERNEL_RUN_TYPE\"),\n              \"gpu_name\": torch.cuda.get_device_name(0), \"gpu_uuid\": uuid,\n              \"source_commit\": SOURCE, \"source_freeze_sha256\": FREEZE,\n              \"h19_manifest_sha256\": H19_MANIFEST, \"adapter_manifest_sha256\": ADAPTER_MANIFEST,\n              \"earlier_fp16_replicated_not_overwritten\": True,\n              \"earlier_int8_unknown\": True, \"earlier_nf4_not_started\": True,\n              \"batch_profile_order\": list(MODES),\n              \"no_adoption\": True, \"training_runs\": 0}\n    atomic_json(output / \"assigned-hardware.json\", report)\n    return report\n\n\ndef run(root=ROOT):\n    source = prepare(root)\n    results = root / \"results\"\n    results.mkdir()\n    batch = {\"schema\": \"hala.h22.batch_test_triplet.v1\",\n             \"status\": \"preparing\", \"source_commit\": SOURCE,\n             \"earlier_fp16_archive_sha256\": EARLIER_FP16_ARCHIVE,\n             \"earlier_int8_outcome\": \"unknown\", \"earlier_nf4_outcome\": \"not_started\",\n             \"profiles\": [], \"training_runs\": 0, \"model_adopted\": False}\n    atomic_json(results / \"batch-status.json\", batch)\n    assigned = qualify(source, results)\n    batch[\"assigned_hardware\"] = assigned\n    batch[\"status\"] = \"running\"\n    atomic_json(results / \"batch-status.json\", batch)\n    env = dict(os.environ, PYTHONIOENCODING=\"utf-8\", CUBLAS_WORKSPACE_CONFIG=\":4096:8\")\n    for idx, mode in enumerate(MODES):\n        if subprocess.check_output([\"git\", \"status\", \"--porcelain\", \"--untracked-files=all\"],\n                                   cwd=source, text=True).strip():\n            raise RuntimeError(\"Scientific source changed during batch\")\n        if subprocess.check_output([\"git\", \"rev-parse\", \"HEAD\"], cwd=source, text=True).strip() != SOURCE:\n            raise RuntimeError(\"Scientific source commit changed during batch\")\n        # Observe the physical UUID again before every new process allocation.\n        import torch\n        from evaluation.development_precision import resolve_device_uuid\n        if resolve_device_uuid(torch) != assigned[\"gpu_uuid\"]:\n            raise RuntimeError(\"Physical GPU changed during TEST batch\")\n        name = f\"profile-{idx}-{mode}-test\"\n        folder = results / name\n        folder.mkdir()\n        args = [sys.executable, \"-u\", \"-m\", \"evaluation.development_precision\",\n                \"--mode\", mode, \"--source-freeze-sha256\", FREEZE,\n                \"--bundle\", str(root / \"frozen\"),\n                \"--manifest-sha256\", H19_MANIFEST, \"--output\", str(folder / \"run\"),\n                \"--font-path\", \"/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf\",\n                \"--source-commit\", SOURCE, \"--provider\", \"kaggle\", \"--split\", \"test\",\n                \"--adapter\", str(root / \"retained-adapter\"),\n                \"--adapter-manifest-sha256\", ADAPTER_MANIFEST,\n                \"--adapter-identity\", json.dumps(ADAPTER_IDENTITY, sort_keys=True)]\n        record = {\"mode\": mode, \"split\": \"test\", \"status\": \"running\",\n                  \"started_unix\": time.time(), \"gpu_uuid\": assigned[\"gpu_uuid\"]}\n        batch[\"profiles\"].append(record)\n        atomic_json(results / \"batch-status.json\", batch)\n        try:\n            with (folder / \"profile.log\").open(\"wb\") as log:\n                process = subprocess.run(args, cwd=source, env=env, stdout=log,\n                                         stderr=subprocess.STDOUT, timeout=3700)\n            record[\"status\"] = \"finished\" if process.returncode == 0 else \"failed\"\n            record[\"returncode\"] = process.returncode\n        except subprocess.TimeoutExpired:\n            record[\"status\"] = \"timeout\"\n        except Exception as error:\n            record[\"status\"] = \"launcher_error\"\n            record[\"error_type\"] = type(error).__name__\n        result = folder / \"run\" / \"result.json\"\n        if result.is_file():\n            content = result.read_bytes()\n            record[\"result_sha256\"] = digest(content)\n            measured = json.loads(content)\n            record[\"result_status\"] = measured.get(\"status\")\n            record[\"complete_tasks\"] = measured.get(\"summary\", {}).get(\"complete\")\n            if measured.get(\"gpu_uuid\") != assigned[\"gpu_uuid\"]:\n                record[\"status\"] = \"physical_device_mismatch\"\n        record[\"finished_unix\"] = time.time()\n        atomic_json(folder / \"profile-status.json\", record)\n        archive = results / (name + \"-evidence.zip\")\n        with zipfile.ZipFile(archive, \"x\", compression=zipfile.ZIP_DEFLATED) as z:\n            for path in sorted(folder.rglob(\"*\")):\n                if path.is_file():\n                    if path.is_symlink() or path.stat().st_size > 8 * 1024**2:\n                        raise ValueError(\"Evidence file exceeds bound; preserve raw directory\")\n                    z.write(path, path.relative_to(folder).as_posix())\n            z.write(results / \"assigned-hardware.json\", \"assigned-hardware.json\")\n            z.write(Path(__file__), \"h22_batch_operator.py\")\n        if archive.stat().st_size > 16 * 1024**2:\n            raise ValueError(\"Evidence ZIP exceeds limit; raw directory preserved\")\n        record[\"archive_sha256\"] = digest(archive.read_bytes())\n        record[\"archive_bytes\"] = archive.stat().st_size\n        atomic_json(results / \"batch-status.json\", batch)\n        print(\"HALA_BATCH_PROFILE:\", json.dumps(record, sort_keys=True), flush=True)\n    batch[\"status\"] = (\"completed_unqualified\" if all(\n        x[\"status\"] == \"finished\" and x.get(\"result_status\") == \"measured\"\n        for x in batch[\"profiles\"]) else \"completed_with_blockers\")\n    atomic_json(results / \"batch-status.json\", batch)\n    print(\"HALA_BATCH_FINISHED:\", json.dumps(batch, sort_keys=True), flush=True)\n    return batch\n\n\nif __name__ == \"__main__\":\n    run()\n"
expected = "44ace0839973b1f308f4bcfc46e864e83074bed6c17805e79edf9a940e72f84e"
assert hashlib.sha256(data.encode('utf-8')).hexdigest() == expected
path = Path('/kaggle/working/hala-h22-operator.py')
assert not path.exists(), 'Do not overwrite existing operator source'
path.write_bytes(data.encode('utf-8'))
print('OPERATOR_SHA256', expected)
subprocess.run([sys.executable, '-u', str(path)], check=True,
               timeout=3*3700+1800)
print('NO PUBLIC RESULT CLAIM UNTIL VERSION OUTPUT RETRIEVAL AND INDEPENDENT AUDIT')
